In [ ]:
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null
!pip install -q onnxruntime==1.24.4 2>/dev/null
!pip install -q onnx-tool==1.0.1 2>/dev/null
!pip install -q jax2onnx==0.13.0 2>/dev/null
!pip install matplotlib 2>/dev/null

In [ ]:
# ── Cell 1: Imports ve yardımcı fonksiyonlar ───────────────────────────────
import os, re, json, shutil, math
from collections import Counter

import numpy as np
import pandas as pd
import jax.numpy as jnp
import jax
import onnx
import onnx.numpy_helper
from jax2onnx import to_onnx
import onnxruntime
import kaggle_benchmarks as kbench

import sys
sys.path.append("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils")
from neurogolf_utils import (
    verify_subset, score_network, run_network,
    load_examples, convert_to_numpy, convert_from_numpy, verify_network,
)

if not hasattr(onnxruntime, "ONNXRuntimeError"):
    onnxruntime.ONNXRuntimeError = Exception

print(f"Devices: {jax.devices()}")


def extract_python_code(text: str) -> str:
    """```python ... ``` bloğunu ya da ham kodu döndürür."""
    if not isinstance(text, str):
        return ""
    match = re.search(r'```python(.*?)```', text, re.DOTALL)
    return match.group(1).strip() if match else text.replace('```', '').strip()


def compile_jax_to_onnx(code_str: str, onnx_filepath: str) -> tuple[bool, str]:
    """JAX solve() fonksiyonunu ONNX dosyasına derler."""
    namespace = {}
    try:
        exec(code_str, globals(), namespace)
        solve_func = namespace.get('solve')
        if not callable(solve_func):
            return False, "'solve' adında çağrılabilir bir fonksiyon bulunamadı."
        dummy_input = jnp.zeros((1, 10, 30, 30), dtype=jnp.float32)
        to_onnx(solve_func, [dummy_input], return_mode="file", output_path=onnx_filepath)
        return True, ""
    except Exception as e:
        return False, str(e)


def compile_onnx_direct(code_str: str, onnx_filepath: str) -> tuple[bool, str]:
    """build_model() fonksiyonunu çalıştırarak ONNX dosyası üretir."""
    namespace = {}
    _globals = {**globals(), 'onnx': onnx, 'np': np, 'numpy': np,
                'math': math, 'onnx_helper': onnx.helper,
                'TensorProto': onnx.TensorProto}
    try:
        exec(code_str, _globals, namespace)
        build_func = namespace.get('build_model')
        if not callable(build_func):
            return False, "'build_model' adında çağrılabilir bir fonksiyon bulunamadı."
        model = build_func()
        onnx.save(model, onnx_filepath)
        return True, ""
    except Exception as e:
        return False, str(e)


def fix_io_names(model: onnx.ModelProto) -> onnx.ModelProto:
    """ONNX grafının I/O isimlerini 'input'/'output' olarak düzeltir."""
    graph = model.graph
    old_inp = graph.input[0].name
    old_out = graph.output[0].name
    if old_inp != "input":
        graph.input[0].name = "input"
        for node in graph.node:
            for i, n in enumerate(node.input):
                if n == old_inp:
                    node.input[i] = "input"
    if old_out != "output":
        graph.output[0].name = "output"
        for node in graph.node:
            for i, n in enumerate(node.output):
                if n == old_out:
                    node.output[i] = "output"
    for node in graph.node:
        if node.output:
            node.name = node.output[0]
    return model

In [ ]:
# ── Cell 2: Gözlemci ──────────────────────────────────────────────────────
class Gozlemci:
    """ARC görevi hakkında LLM'e verilmek üzere yapısal bilgi çıkarır."""

    def analyze(self, examples: dict) -> dict:
        train = examples.get('train', [])
        test  = examples.get('test',  [])
        all_pairs = train + test

        if not all_pairs:
            return {'example_summary': 'Örnek bulunamadı.'}

        # Grid boyutları
        input_sizes  = [(len(p['input']),  len(p['input'][0]))  for p in all_pairs]
        output_sizes = [(len(p['output']), len(p['output'][0])) for p in all_pairs]

        # Renk paleti
        all_colors: set[int] = set()
        for p in all_pairs:
            for row in p['input']:  all_colors.update(row)
            for row in p['output']: all_colors.update(row)

        # Arka plan rengi (ilk train örneğinin girdisinde en sık tekrarlayan)
        flat = [c for row in all_pairs[0]['input'] for c in row]
        background_color = Counter(flat).most_common(1)[0][0]

        # Boyut değişimi var mı?
        size_changes   = [(o[0]-i[0], o[1]-i[1]) for i, o in zip(input_sizes, output_sizes)]
        has_size_change = any(dc != (0, 0) for dc in size_changes)

        # Basit dönüşüm sezgileri
        is_color_mapping = not has_size_change

        # LLM'e özet metin
        unique_in  = sorted(set(input_sizes))
        unique_out = sorted(set(output_sizes))
        summary = (
            f"- {len(train)} eğitim, {len(test)} test, {len(examples.get('arc-gen',[]))} arc-gen çifti\n"
            f"- Giriş boyutları (satır×sütun): {unique_in}\n"
            f"- Çıkış boyutları: {unique_out}\n"
            f"- Boyut değişimi: {'VAR' if has_size_change else 'YOK (giriş=çıkış)'}\n"
            f"- Kullanılan renkler (kanal indeksleri): {sorted(all_colors)}\n"
            f"- Arka plan rengi (baskın): {background_color}\n"
            f"- Sezgi: {'Renk eşlemesi olabilir' if is_color_mapping else 'Uzamsal/yapısal dönüşüm'}"
        )

        return {
            'input_sizes':      input_sizes,
            'output_sizes':     output_sizes,
            'color_palette':    sorted(all_colors),
            'background_color': background_color,
            'has_size_change':  has_size_change,
            'is_color_mapping': is_color_mapping,
            'example_summary':  summary,
        }

In [ ]:
# ── Cell 3: Sentezci ──────────────────────────────────────────────────────
class Sentezci:
    """Gözlemci çıktısı + geri bildirimle LLM için hedefli prompt üretir."""

    _STATIC_CONSTRAINTS = (
        "CRITICAL CONSTRAINTS:\n"
        "1. STATIC SHAPES: input is float32 [1,10,30,30]. Output MUST be EXACTLY [1,10,30,30].\n"
        "2. FORBIDDEN OPS:\n"
        "   - NO boolean indexing  → creates banned NonZero/Compress ops\n"
        "   - NO jnp.where, jnp.nonzero, jnp.argwhere\n"
        "   - NO dynamic slices based on data values\n"
        "   - NO Loop / Scan / Unique / Script / Function ONNX ops\n"
        "3. ALGEBRAIC MASKING: for 'if cond then A else B' use: (mask*A) + ((1-mask)*B)\n"
        "4. NO RANK REDUCTION: use grid[:,0:1,:,:] not grid[0]\n"
        "5. BACKGROUND: cells outside the active region must be 0.0 across all 10 channels.\n"
    )

    _JAX_SYSTEM = (
        "You are an ML engineer specializing in JAX-to-ONNX static graph compilation.\n"
        "Write a Python function named `solve(grid)` using JAX only.\n"
        "Allowed: jax.numpy ops, jax.scipy.signal.convolve2d, jnp.roll, static padding/slicing.\n"
    )

    _ONNX_DIRECT_SYSTEM = (
        "You are an ML engineer building minimal ONNX graphs directly with onnx.helper.\n"
        "Previous JAX approaches failed or scored too low. Build the graph from scratch.\n\n"
        "Write a function `build_model()` that returns an onnx.ModelProto.\n"
        "Use ONLY these ops: Conv, MatMul, Add, Sub, Mul, Relu, Sigmoid, Transpose, Reshape,\n"
        "  Concat, Slice (with STATIC start/end/axes/steps as inputs), Pad.\n"
        "Input name must be 'input', output name 'output', both shape [1,10,30,30] float32.\n\n"
        "Minimal example skeleton:\n"
        "```python\n"
        "import onnx, onnx.helper, onnx.numpy_helper, numpy as np\n\n"
        "def build_model():\n"
        "    W = np.zeros((10, 10, 1, 1), dtype=np.float32)\n"
        "    # fill W with the transformation weights ...\n"
        "    w_init = onnx.numpy_helper.from_array(W, name='W')\n"
        "    x = onnx.helper.make_tensor_value_info('input',  onnx.TensorProto.FLOAT, [1,10,30,30])\n"
        "    y = onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1,10,30,30])\n"
        "    conv = onnx.helper.make_node('Conv', ['input','W'], ['output'],\n"
        "                                kernel_shape=[1,1], pads=[0,0,0,0])\n"
        "    graph = onnx.helper.make_graph([conv], 'g', [x], [y], [w_init])\n"
        "    return onnx.helper.make_model(graph, ir_version=10,\n"
        "                                  opset_imports=[onnx.helper.make_opsetid('',10)])\n"
        "```\n"
    )

    def build_prompt(
        self,
        task_analysis: dict,
        examples: dict,
        attempt: int,
        feedback: str = "",
        prev_code: str = "",
        strategy: str = "jax",
    ) -> str:
        is_optimization = "COST SUMMARY" in feedback and attempt > 1

        if strategy == "onnx_direct":
            prompt = self._ONNX_DIRECT_SYSTEM + "\n" + self._STATIC_CONSTRAINTS
        else:
            prompt = self._JAX_SYSTEM + "\n" + self._STATIC_CONSTRAINTS

        # Task analysis context
        prompt += "\nTASK ANALYSIS:\n" + task_analysis.get('example_summary', '') + "\n"

        # Feedback from previous attempt
        if feedback and attempt > 1:
            if is_optimization:
                prompt += (
                    f"\n==[ ATTEMPT {attempt-1}: LOGIC CORRECT BUT TOO EXPENSIVE ]==\n"
                    "Your model was logically correct but uses too many parameters/memory.\n"
                    "GOAL: Re-implement the same logic with FEWER parameters.\n"
                    "Prefer: 1×1 convolutions, fewer channels, no redundant ops.\n"
                    f"\n{feedback}\n"
                )
            else:
                prompt += (
                    f"\n==[ ATTEMPT {attempt-1} FAILED ]==\n"
                    f"{feedback}\n"
                )
            if prev_code:
                prompt += f"\nYOUR PREVIOUS CODE:\n```python\n{prev_code}\n```\n"

        # Examples
        prompt += "\nTASK EXAMPLES:\n"
        for i, ex in enumerate(examples.get('train', [])):
            prompt += f"Example {i+1}:\nInput:\n{ex['input']}\nOutput:\n{ex['output']}\n\n"

        if strategy == "onnx_direct":
            prompt += "Output ONLY the raw Python code block starting with `def build_model():`."
        else:
            prompt += "Output ONLY the raw Python code block starting with `def solve(grid):`."

        return prompt

In [ ]:
# ── Cell 4: Hakem ─────────────────────────────────────────────────────────
class Hakem:
    """
    ONNX modelini değerlendirir:
    - Mantık doğruluğu (tüm örünek subsetleri)
    - Bellek/parametre maliyeti analizi
    - LLM'i optimize etmeye yönlendiren detaylı geri bildirim
    """

    def evaluate(self, model: onnx.ModelProto, examples: dict, session) -> dict:
        result = dict(
            is_correct=False, score=0.0,
            agi_pass=0, agi_fail=0,
            gen_pass=0, gen_fail=0,
            memory=0, params=0,
            feedback="", failure_analysis="", cost_analysis="",
        )

        # 1. Doğruluk kontrolü (profiling aktifken)
        agi_r, agi_w, _ = verify_subset(
            session, examples.get("train", []) + examples.get("test", [])
        )
        gen_r, gen_w, _ = verify_subset(session, examples.get("arc-gen", []))

        result["agi_pass"], result["agi_fail"] = agi_r, agi_w
        result["gen_pass"], result["gen_fail"] = gen_r, gen_w

        # 2. Profil dökümü (her durumda)
        trace_path = session.end_profiling()

        total_wrong = agi_w + gen_w

        if total_wrong > 0:
            result["failure_analysis"] = self._analyze_failures(session, examples)
            result["feedback"] = (
                f"LOGIC ERROR: {total_wrong} example(s) failed "
                f"({agi_w} ARC-AGI, {gen_w} ARC-GEN).\n"
                + result["failure_analysis"]
            )
            return result

        # 3. Skor hesapla
        memory, params = score_network(model, trace_path)
        if memory is None or params is None:
            result["feedback"] = (
                "Static shape/profile error. "
                "Check for dynamic typing, disallowed ops, or mismatched node names."
            )
            return result

        result["memory"]  = memory
        result["params"]  = params
        result["score"]   = round(max(1.0, 25.0 - math.log(max(1.0, memory + params))), 3)
        result["is_correct"] = True

        # 4. Maliyet analizi + optimizasyon önerileri
        result["cost_analysis"] = self._analyze_graph_cost(model, memory, params)
        result["feedback"]      = result["cost_analysis"]
        return result

    # ── Yardımcı: mantık hatası analizi ──────────────────────────────────
    def _analyze_failures(self, session, examples: dict) -> str:
        lines: list[str] = []
        subsets = [
            ("train",   examples.get("train", [])),
            ("test",    examples.get("test",  [])),
            ("arc-gen", examples.get("arc-gen", [])[:10]),
        ]
        for label, pairs in subsets:
            for i, ex in enumerate(pairs):
                bench = convert_to_numpy(ex)
                if bench is None:
                    continue
                try:
                    user_out = run_network(session, bench["input"])
                    if np.array_equal(user_out, bench["output"]):
                        continue
                    diff = bench["output"] - user_out
                    wrong_chs = np.where(
                        np.any(diff != 0, axis=(0, 2, 3))
                    )[0].tolist()
                    lines.append(f"[{label}#{i}] Wrong channels: {wrong_chs}")
                    for ch in wrong_chs[:4]:
                        exp_s = float(bench["output"][0, ch].sum())
                        act_s = float(user_out[0, ch].sum())
                        lines.append(f"  ch{ch}: expected_sum={exp_s:.0f}, actual_sum={act_s:.0f}")
                except Exception as exc:
                    lines.append(f"[{label}#{i}] Runtime error: {str(exc)[:120]}")
                if len(lines) >= 24:
                    lines.append("... (truncated)")
                    return "\n".join(lines)
        return "\n".join(lines) if lines else "Failure pattern unknown."

    # ── Yardımcı: ONNX graf maliyet analizi ──────────────────────────────
    def _analyze_graph_cost(self, model: onnx.ModelProto, memory: int, params: int) -> str:
        total = memory + params
        score = max(1.0, 25.0 - math.log(max(1.0, total)))

        lines = [
            "=== COST SUMMARY ===",
            f"Memory: {memory:,} bytes | Params: {params:,} | Total: {total:,}",
            f"Current score: {score:.3f}/25",
            f"To reach 20pts: total ≤ {int(math.exp(5)):,}  "
            f"| 22pts: ≤ {int(math.exp(3)):,}  "
            f"| 25pts: ≤ 1",
            "",
        ]

        # Per-node parametre dağılımı
        init_map = {init.name: init for init in model.graph.initializer}
        node_costs: list[tuple[int, str, list[str]]] = []
        for node in model.graph.node:
            n_params = 0
            details: list[str] = []
            for inp in node.input:
                if inp in init_map:
                    init = init_map[inp]
                    shape = list(init.dims)
                    cnt   = math.prod(shape) if shape else 0
                    n_params += cnt
                    details.append(f"{inp}:{shape}={cnt}p")
            if n_params > 0:
                node_costs.append((n_params, node.op_type, details))

        node_costs.sort(reverse=True)
        if node_costs:
            lines.append("TOP EXPENSIVE NODES:")
            for cost, op, details in node_costs[:5]:
                lines.append(f"  {op}: {', '.join(details)}  →  {cost} params")
                # Op-spesifik öneriler
                if op == "Conv":
                    for d in details:
                        m = re.search(r'\[(\d+),(\d+),(\d+),(\d+)\]', d)
                        if m:
                            co, ci, kh, kw = int(m[1]), int(m[2]), int(m[3]), int(m[4])
                            if kh > 1 or kw > 1:
                                lines.append(
                                    f"    → 1×1 kernel alternative: {co*ci} params "
                                    f"(saves {co*ci*kh*kw - co*ci})"
                                )
                            if co > 10 or ci > 10:
                                lines.append(
                                    f"    → Channel count exceeds 10 (ARC limit). "
                                    "Consider reducing to exactly 10 in/out channels."
                                )
            lines.append("")

        # Seyreklik kontrolü
        total_w, zero_w = 0, 0
        for init in model.graph.initializer:
            try:
                arr = onnx.numpy_helper.to_array(init).ravel()
                total_w += arr.size
                zero_w  += int((arr == 0).sum())
            except Exception:
                pass
        if total_w > 0 and zero_w / total_w > 0.5:
            pct = zero_w / total_w * 100
            lines.append(
                f"SPARSITY: {pct:.0f}% of weights are zero. "
                "Consider using sparse_initializer or eliminating zero channels."
            )

        # Genel optimizasyon önerileri
        lines += [
            "",
            "OPTIMIZATION SUGGESTIONS:",
        ]
        if params > 1000:
            lines.append(
                "  • This transformation may be expressible with a single 1×1 Conv "
                "(10×10=100 params max)."
            )
        if len(list(model.graph.node)) > 3:
            lines.append(
                "  • Multiple nodes detected — try merging into fewer ops."
            )
        if memory > params * 10:
            lines.append(
                "  • Memory footprint is large relative to params. "
                "Check for large intermediate tensors."
            )
        lines.append(
            "  • Use float16 or int8 precision where possible to halve/quarter memory."
        )
        return "\n".join(lines)

In [ ]:
# ── Cell 5: TaskSolver ────────────────────────────────────────────────────
class TaskSolver:
    """
    Gözlemci → Sentezci → LLM → Derleyici → Hakem döngüsünü yönetir.
    Tüm ara ve nihai çıktıları disk'e kaydeder.
    """

    def __init__(
        self,
        llm,
        min_score: float = 15.0,
        max_attempts: int = 8,
        output_dir: str = "/kaggle/working/results",
    ):
        self.llm         = llm
        self.min_score   = min_score
        self.max_attempts = max_attempts
        self.output_dir  = output_dir
        self.gozlemci    = Gozlemci()
        self.sentezci    = Sentezci()
        self.hakem       = Hakem()

    def solve(self, task_num: int, examples: dict) -> dict:
        task_id  = f"task{task_num:03d}"
        task_dir = os.path.join(self.output_dir, task_id)
        os.makedirs(task_dir, exist_ok=True)

        # 1. Gözlem
        task_analysis = self.gozlemci.analyze(examples)

        best_result   = {"is_correct": False, "score": 0.0}
        best_onnx_src = None
        feedback      = ""
        prev_code     = ""

        for attempt in range(1, self.max_attempts + 1):
            strategy   = "jax" if attempt <= 3 else "onnx_direct"
            onnx_path  = os.path.join(task_dir, f"attempt_{attempt}.onnx")
            trace_pfx  = os.path.join(task_dir, f"attempt_{attempt}_trace")

            # 2. Prompt üret
            prompt = self.sentezci.build_prompt(
                task_analysis, examples, attempt,
                feedback, prev_code, strategy,
            )

            # 3. LLM çağrısı
            try:
                raw_response = (
                    self.llm.prompt(prompt)
                    if hasattr(self.llm, 'prompt')
                    else self.llm.generate(prompt)
                )
            except Exception as exc:
                feedback = f"LLM call failed: {exc}"
                print(f"[{task_id}] Attempt {attempt}: LLM error — {exc}")
                continue

            # Ham kayıtlar
            self._save(task_dir, f"attempt_{attempt}_prompt.txt",   prompt)
            self._save(task_dir, f"attempt_{attempt}_response.txt", raw_response)

            # 4. Kod çıkar ve kaydet
            code      = extract_python_code(raw_response)
            prev_code = code
            self._save(task_dir, f"attempt_{attempt}_code.py", code)

            # 5. ONNX derle
            compile_fn = compile_jax_to_onnx if strategy == "jax" else compile_onnx_direct
            success, err_msg = compile_fn(code, onnx_path)
            if not success:
                feedback = f"Compile Error ({strategy}): {err_msg}"
                print(f"[{task_id}] Attempt {attempt} ({strategy}): compile fail")
                continue

            # 6. I/O isim düzelt ve yükle
            try:
                model = onnx.load(onnx_path)
                model = fix_io_names(model)
                onnx.save(model, onnx_path)
            except Exception as exc:
                feedback = f"ONNX load/fix error: {exc}"
                print(f"[{task_id}] Attempt {attempt}: ONNX fix error — {exc}")
                continue

            # 7. OnnxRuntime oturumu (profiling açık)
            try:
                opts = onnxruntime.SessionOptions()
                opts.enable_profiling            = True
                opts.graph_optimization_level    = (
                    onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
                )
                opts.profile_file_prefix = trace_pfx
                session = onnxruntime.InferenceSession(
                    model.SerializeToString(), opts
                )
            except onnxruntime.ONNXRuntimeError as exc:
                feedback = f"ONNX session error: {exc}"
                print(f"[{task_id}] Attempt {attempt}: session error — {exc}")
                continue

            # 8. Hakem değerlendirmesi (end_profiling içinde çağrılır)
            eval_result = self.hakem.evaluate(model, examples, session)

            # Hakem raporunu kaydet
            self._save_json(task_dir, f"attempt_{attempt}_result.json", eval_result)

            print(
                f"[{task_id}] Attempt {attempt} ({strategy}): "
                f"correct={eval_result['is_correct']}, "
                f"score={eval_result['score']}, "
                f"agi={eval_result['agi_pass']}/{eval_result['agi_pass']+eval_result['agi_fail']}, "
                f"gen={eval_result['gen_pass']}/{eval_result['gen_pass']+eval_result['gen_fail']}"
            )

            feedback = eval_result["feedback"]

            # En iyi modeli kaydet
            if eval_result["score"] > best_result["score"]:
                best_result   = eval_result
                best_onnx_src = onnx_path

            # Hedef skora ulaşıldı mı?
            if eval_result["is_correct"] and eval_result["score"] >= self.min_score:
                break

        # En iyi ONNX'i kalıcı yere kopyala
        best_onnx_path = None
        if best_onnx_src:
            best_onnx_path = os.path.join(task_dir, f"{task_id}_best.onnx")
            shutil.copy2(best_onnx_src, best_onnx_path)

        summary = {
            "task_id":       task_id,
            "is_correct":    best_result["is_correct"],
            "best_score":    best_result["score"],
            "attempts_used": attempt,
            "onnx_path":     best_onnx_path,
            "task_analysis": task_analysis,
        }
        self._save_json(task_dir, "summary.json", summary)
        return summary

    # ── Disk kayıt yardımcıları ───────────────────────────────────────────
    @staticmethod
    def _save(task_dir: str, filename: str, content: str) -> None:
        with open(os.path.join(task_dir, filename), "w", encoding="utf-8") as f:
            f.write(content)

    @staticmethod
    def _save_json(task_dir: str, filename: str, data) -> None:
        with open(os.path.join(task_dir, filename), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, default=str)

In [ ]:
# ── Cell 6: Çalıştırma (kbench) ───────────────────────────────────────────

# ── Parametreler ──────────────────────────────────────────────────────────
MIN_SCORE    = 15.0   # Bu eşiğe ulaşınca görev tamamlanmış sayılır
MAX_ATTEMPTS = 8      # Görev başına maksimum LLM denemesi
OUTPUT_DIR   = "/kaggle/working/results"
TASK_NUMBERS = [4]    # Çözülecek görev numaraları (1-400)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LLM listesi ────────────────────────────────────────────────────────────
try:
    available_models = [
        kbench.llms['google/gemini-3.1-pro-preview'],
        # kbench.llms['anthropic/claude-opus-4-7@default'],
        # kbench.llms['openai/gpt-5.4-2026-03-05'],
    ]
except KeyError as e:
    print(f"Model bulunamadı: {e}. Mevcut ilk model kullanılıyor.")
    available_models = list(kbench.llms.values())[:1]

# ── kbench görev sarmalayıcısı ─────────────────────────────────────────────
@kbench.task(name="arc_solver", store_task=False)
def arc_task(llm, task_num: int, examples: dict) -> dict:
    solver = TaskSolver(
        llm=llm,
        min_score=MIN_SCORE,
        max_attempts=MAX_ATTEMPTS,
        output_dir=OUTPUT_DIR,
    )
    return solver.solve(task_num, examples)

# ── Değerlendirme verisi ───────────────────────────────────────────────────
evaluation_data = []
for t_num in TASK_NUMBERS:
    ex = load_examples(t_num)
    if ex:
        evaluation_data.append({"task_num": t_num, "examples": ex})

df      = pd.DataFrame(evaluation_data)
n_total = len(available_models) * len(df)

print(f"Değerlendirme: {len(available_models)} model × {len(df)} görev = {n_total} koşum")
for m in available_models:
    print(f"  • {getattr(m, 'name', str(m))}")

# ── Çalıştır ───────────────────────────────────────────────────────────────
runs = arc_task.evaluate(
    stop_condition=lambda runs: len(runs) == n_total,
    max_attempts=1,
    retry_delay=15,
    llm=available_models,
    evaluation_data=df,
    n_jobs=1,
)

# ── Sonuçlar ───────────────────────────────────────────────────────────────
eval_df = runs.as_dataframe()

if not eval_df.empty and 'result' in eval_df.columns:
    metrics_df = pd.json_normalize(eval_df['result'])
    final_df   = pd.concat(
        [eval_df.drop(columns=['result']), metrics_df.set_index(eval_df.index)],
        axis=1,
    )
    final_df['model_name'] = final_df['llm'].apply(
        lambda obj: getattr(obj, 'name', str(obj))
    )

    show_cols = [c for c in
                 ['model_name', 'task_id', 'is_correct', 'best_score', 'attempts_used']
                 if c in final_df.columns]

    print("\n" + "="*60)
    print("SONUÇLAR")
    print("="*60)
    print(final_df[show_cols].to_string(index=False))

    if 'best_score' in final_df.columns:
        summary = final_df.groupby('model_name').agg(
            Dogruluk=('is_correct', 'mean'),
            Ort_Skor=('best_score', 'mean'),
            Toplam_Skor=('best_score', 'sum'),
        )
        print("\nMODEL PERFORMANS ÖZETİ:")
        print(summary)
else:
    print("Sonuç hesaplanamadı.")